# Experiment 15 — RWKV (Receptance Weighted Key Value)

Second notebook in the subquadratic-attention track (see experiment 14's intro for the
full framing: fixed-size recurrent state instead of an O(T²) attention cache, same
`K`-pair multi-query associative recall test used across 14-17 so results are directly
comparable).

**RWKV** (Peng et al., 2023, "RWKV: Reinventing RNNs for the Transformer Era") takes a
different route to linear-time sequence modeling than Mamba's state-space formulation.
It's built around one recurrence per channel called **WKV**: an exponentially-decaying
weighted sum of all past values, with a learned per-channel decay rate `w` and a special
"bonus" `u` that lets the *current* token count extra toward its own output (without `u`,
a token's own contribution would always be diluted by the running sum). Unlike a plain
weighted average, RWKV's decay rate is a fixed learned parameter per channel (not
input-dependent the way Mamba's `Δ` is) — it's a simpler, more RNN-like mechanism, which
is part of the point: RWKV is explicitly designed to be trained like a Transformer
(parallelizable across time during training) but run like an RNN (constant memory,
one step at a time) at inference.

## The WKV recurrence

For a single channel, the quantity being computed at each step `t` is:

```
wkv_t = ( sum_{i<t} exp(-(t-1-i)w + k_i) v_i  +  exp(u + k_t) v_t )
        -----------------------------------------------------------
        ( sum_{i<t} exp(-(t-1-i)w + k_i)        +  exp(u + k_t) )
```

i.e. a weighted average of every value seen so far, where older values decay by `exp(-w)`
per step, and the current token gets a one-off `exp(u)` bonus weight instead of the decay
schedule. Computed naïvely this over/underflows almost immediately (the exponents grow
without bound), so this notebook uses the actual numerically-stable recurrent form from
the official RWKV-4 implementation: three running quantities `(aa, bb, pp)` — numerator,
denominator, and a running max exponent used to keep every `exp(...)` argument ≤ 0.

Before `k`, `v`, and the output gate `r` (**r**eceptance) are computed, RWKV mixes each
token with the *previous* token via a learned interpolation (“token shift”,
`x·μ + x_prev·(1-μ)`, a separate `μ` for each of k/v/r) — cheap substitute for a
convolution that still lets each position see one step of local context. A
**channel-mixing** block (a small squared-ReLU feed-forward, also token-shifted) follows
the time-mixing block, mirroring the attention+FFN structure of a Transformer block.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# --- same MQAR task as experiment 14, copied rather than imported (one-file-per-notebook rule) ---
K = 8
M = 4
SEP = 1
KEY0 = 2
VAL0 = 2 + K
VOCAB = 2 + 2 * K


def make_batch(batch_size, device="cpu"):
    value_assignment = torch.argsort(torch.rand(batch_size, K), dim=1)
    order = torch.argsort(torch.rand(batch_size, K), dim=1)
    key_ids = KEY0 + order
    val_ids = VAL0 + torch.gather(value_assignment, 1, order)
    context = torch.stack([key_ids, val_ids], dim=2).reshape(batch_size, 2 * K)
    sep = torch.full((batch_size, 1), SEP, dtype=torch.long)
    q_key_identity = torch.randint(0, K, (batch_size, M))
    query_tokens = KEY0 + q_key_identity
    query_labels = torch.gather(value_assignment, 1, q_key_identity)
    seq = torch.cat([context, sep, query_tokens], dim=1)
    return seq.to(device), query_labels.to(device)


seq, labels = make_batch(1)
print("one example sequence:", seq[0].tolist())
print("correct value-class for each query:", labels[0].tolist())


device: cuda
one example sequence: [6, 10, 7, 15, 8, 11, 4, 14, 5, 16, 2, 12, 9, 17, 3, 13, 1, 2, 6, 4, 3]
correct value-class for each query: [2, 0, 4, 3]


In [2]:
class RWKVTimeMix(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.time_decay = nn.Parameter(torch.zeros(d_model))   # w, used as -exp(w): per-channel decay
        self.time_first = nn.Parameter(torch.zeros(d_model))   # u, the "bonus" for the current token
        self.mu_k = nn.Parameter(torch.rand(d_model))
        self.mu_v = nn.Parameter(torch.rand(d_model))
        self.mu_r = nn.Parameter(torch.rand(d_model))
        self.key = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_model, d_model)
        self.receptance = nn.Linear(d_model, d_model)
        self.output = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, C = x.shape
        x_prev = F.pad(x, (0, 0, 1, 0))[:, :T, :]          # token-shift: x_{t-1}, zero at t=0
        xk = x * self.mu_k + x_prev * (1 - self.mu_k)
        xv = x * self.mu_v + x_prev * (1 - self.mu_v)
        xr = x * self.mu_r + x_prev * (1 - self.mu_r)
        k = self.key(xk)
        v = self.value(xv)
        r = torch.sigmoid(self.receptance(xr))
        w = -torch.exp(self.time_decay)   # always negative -> genuine decay
        u = self.time_first

        # numerically-stable WKV recurrence (matches the official RWKV-4 formulation)
        aa = x.new_zeros(B, C)
        bb = x.new_zeros(B, C)
        pp = x.new_full((B, C), -1e30)
        outs = []
        for t in range(T):
            kt, vt = k[:, t, :], v[:, t, :]
            ww = u + kt
            q = torch.maximum(pp, ww)
            e1 = torch.exp(pp - q)
            e2 = torch.exp(ww - q)
            wkv = (e1 * aa + e2 * vt) / (e1 * bb + e2)
            outs.append(wkv)

            ww2 = pp + w
            q2 = torch.maximum(ww2, kt)
            e1b = torch.exp(ww2 - q2)
            e2b = torch.exp(kt - q2)
            aa = e1b * aa + e2b * vt
            bb = e1b * bb + e2b
            pp = q2
        wkv_out = torch.stack(outs, dim=1)
        return self.output(r * wkv_out)


class RWKVChannelMix(nn.Module):
    def __init__(self, d_model, d_hidden=None):
        super().__init__()
        d_hidden = d_hidden or 4 * d_model
        self.mu_k = nn.Parameter(torch.rand(d_model))
        self.mu_r = nn.Parameter(torch.rand(d_model))
        self.key = nn.Linear(d_model, d_hidden)
        self.receptance = nn.Linear(d_model, d_model)
        self.value = nn.Linear(d_hidden, d_model)

    def forward(self, x):
        T = x.shape[1]
        x_prev = F.pad(x, (0, 0, 1, 0))[:, :T, :]
        xk = x * self.mu_k + x_prev * (1 - self.mu_k)
        xr = x * self.mu_r + x_prev * (1 - self.mu_r)
        k = torch.relu(self.key(xk)) ** 2
        r = torch.sigmoid(self.receptance(xr))
        return r * self.value(k)


class RWKVBlock(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.tmix = RWKVTimeMix(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.cmix = RWKVChannelMix(d_model)

    def forward(self, x):
        x = x + self.tmix(self.ln1(x))
        x = x + self.cmix(self.ln2(x))
        return x


## Wiring it into a tiny sequence model

`RWKVBlock` already includes its own residual connections around time-mix and
channel-mix (unlike experiment 14's bare `MambaBlock`, which needed an external residual
wrapper), so the model here is just an embedding, a stack of `RWKVBlock`s, and the same
classifier head / query-position-only loss as experiment 14.

In [3]:
class TokenModelRaw(nn.Module):
    def __init__(self, layer_factory, d_model, n_layers, vocab=VOCAB, n_classes=K):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        self.layers = nn.ModuleList([layer_factory() for _ in range(n_layers)])
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)

    def forward(self, tokens):
        x = self.embed(tokens)
        for layer in self.layers:
            x = layer(x)
        return self.head(self.final_norm(x))


def train_and_eval(model, steps=3000, batch_size=64, lr=3e-3, n_queries=M, log_every=500):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        seq, labels = make_batch(batch_size, device)
        logits = model(seq)[:, -n_queries:, :]
        loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        opt.zero_grad()
        loss.backward()
        opt.step()
        if step % log_every == 0 or step == steps - 1:
            acc = (logits.argmax(-1) == labels).float().mean().item()
            print(f"step {step:4d}  loss {loss.item():.4f}  train_acc {acc:.3f}")

    model.eval()
    with torch.no_grad():
        seq, labels = make_batch(2000, device)
        logits = model(seq)[:, -n_queries:, :]
        test_acc = (logits.argmax(-1) == labels).float().mean().item()
    print(f"\nFINAL TEST ACC: {test_acc:.4f}   (chance = {1/K:.4f})")
    return test_acc


torch.manual_seed(0)
rwkv_model = TokenModelRaw(lambda: RWKVBlock(64), d_model=64, n_layers=2)
n_params = sum(p.numel() for p in rwkv_model.parameters())
print(f"params: {n_params:,}")
test_acc = train_and_eval(rwkv_model)


params: 110,984


step    0  loss 2.3327  train_acc 0.137


step  500  loss 2.0822  train_acc 0.109


step 1000  loss 2.0812  train_acc 0.094


step 1500  loss 1.4533  train_acc 0.418


step 2000  loss 0.0049  train_acc 1.000


step 2500  loss 0.0001  train_acc 1.000


step 2999  loss 0.0002  train_acc 1.000

FINAL TEST ACC: 1.0000   (chance = 0.1250)


## What actually happened

**RWKV solved the task too, with the same shape as Mamba but slower to get there.**
Train accuracy sat at chance (0.09-0.14 against a 0.125 floor) all the way through step
1000, climbed to 0.418 by step 1500, then jumped to 1.000 by step 2000 and held. The first
attempt at this notebook used the same 1500-step budget as experiment 14 and stopped
partway through the climb at 0.373 test accuracy — genuinely still learning, not stuck,
so training was extended to 3000 steps to let it actually finish. Final held-out test
accuracy: **1.0000** on 2,000 fresh examples (chance = 0.1250), with 110,984 parameters
(RWKV's channel-mixing block roughly triples the parameter count next to Mamba's 42k for
the same `d_model`/depth, since it adds a full 4x-widening feed-forward on top of
time-mixing).

**Comparing to experiment 14 so far:** both architectures show the same qualitative
pattern — a long plateau at chance, then a fast climb to perfect accuracy — but RWKV's
transition lands roughly 800-1000 steps later than Mamba's (which finished by step 1200).
Two real architectural differences are plausible causes, neither confirmed in isolation
here: RWKV's decay rate `w` is a fixed learned-once-per-channel constant, while Mamba's
`Δ` is recomputed per token from the input (more directly "selective" about when to
write); and RWKV's local-context mechanism is a single-step token-shift blend, versus
Mamba's 3-wide causal convolution, giving Mamba a slightly wider local window to bind a
key to its value before the recurrence takes over. Both are consistent with RWKV simply
needing more gradient steps to discover the same write-then-recall strategy, not a
capability gap — it reaches the identical 1.0000 ceiling once it gets there.

**What this does and doesn't show:** same caveats as experiment 14 — single seed, toy
scale (`K=8`, `T=21`), sequential Python recurrence rather than a fused kernel. The
question of whether a fixed-size WKV state keeps working as `K` grows past what it can
comfortably hold is not tested here; that capacity question is what motivates the
delta-rule architectures in the next two notebooks.